# CSNAnalysis Tutorial 1: Getting Started

**Dickson Lab, Michigan State University**

---

## What is a Conformation Space Network?

A **Conformation Space Network (CSN)** is a graph-based representation of a molecular free energy
landscape. Each node is a cluster of conformations (e.g. from MSMBuilder or a similar clustering
tool), and each directed edge represents observed transitions between clusters in a molecular
dynamics simulation.

The weight on an edge from state $j$ to state $i$ is the conditional transition probability:

$$t_{ij} = \frac{c_{ij}}{\sum_k c_{kj}}$$

where $c_{ij}$ is the number of observed transitions from $j$ to $i$ in the count matrix.

**This notebook covers:**
1. Loading a count matrix and creating a CSN
2. The three internal representations (count matrix, transition matrix, NetworkX graph)
3. Trimming poorly-sampled or poorly-connected states
4. Computing steady-state weights

## Setup

Install CSNAnalysis with `pip install CSNAnalysis`, then import the dependencies below.

In [ ]:
import numpy as np
import scipy.sparse
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline

from csnanalysis.csn import CSN

## 1. Loading the count matrix

We start from a count matrix — a square matrix where entry $(i, j)$ records the number of
observed transitions from state $j$ to state $i$ in the simulation.

For large systems most pairs of states never communicate directly, so the matrix is
**sparse**: only a small fraction of entries are non-zero.  We load it in scipy's
coordinate (COO) sparse format.

In [ ]:
count_mat = scipy.sparse.load_npz('matrix.npz')

n = count_mat.shape[0]
nnz = len(count_mat.data)
print(f"Matrix size : {n} × {n}")
print(f"Non-zero entries : {nnz}  ({100*nnz/n**2:.3f}% of all entries)")

In [ ]:
# Inspect the first few non-zero entries: (row, column, count)
print("First 10 non-zero entries (to, from, count):")
for r, c, d in zip(count_mat.row[:10], count_mat.col[:10], count_mat.data[:10]):
    print(f"  ({r:4d}, {c:4d}) : {d:.0f}")

## 2. Creating a CSN object

`CSN` accepts a numpy array, a scipy sparse matrix, or a list of lists.
Setting `symmetrize=True` replaces $C$ with $\frac{1}{2}(C + C^T)$, which enforces
detailed balance and is appropriate for equilibrium simulations.

In [ ]:
our_csn = CSN(count_mat, symmetrize=True)
print(f"CSN created with {our_csn.nnodes} nodes")

## 3. The three internal representations

The CSN object stores the data in three complementary forms.

### Count matrix (`our_csn.countmat`)
The original (possibly symmetrized) count matrix in scipy COO format.

In [ ]:
print(our_csn.countmat)

### Transition matrix (`our_csn.transmat`)
Derived from the count matrix by column-normalisation. Each column sums to 1.

In [ ]:
col_sums = our_csn.transmat.sum(axis=0)
print(f"Transition matrix column sums: min = {col_sums.min():.4f}, max = {col_sums.max():.4f}")

### NetworkX graph (`our_csn.graph`)
A directed graph where edge weights are proportional to transition probabilities.
Any NetworkX function can be applied directly.

In [ ]:
# Example: shortest path between two states
path = nx.shortest_path(our_csn.graph, 0, 10)
print(f"Shortest path from state 0 to state 10: {path}")

## 4. Trimming the CSN

Raw count matrices often contain poorly-sampled states that are only weakly connected
to the main network.  `trim()` removes states that:

- are not reachable from the most-sampled node (controlled by `by_outflow`), or
- cannot reach the most-sampled node (controlled by `by_inflow`), or
- have a total count below `min_count`.

Transitions that originally went to trimmed states are reassigned as self-transitions
so that the trimmed transition matrix remains column-stochastic.

In [ ]:
our_csn.trim(by_inflow=True, by_outflow=True, min_count=20)

print(f"Nodes before trimming : {our_csn.nnodes}")
print(f"Nodes after  trimming : {our_csn.trim_nnodes}")
print(f"Retained fraction     : {our_csn.trim_nnodes / our_csn.nnodes:.1%}")

In [ ]:
# The trimmed transition matrix is column-stochastic
col_sums_trim = our_csn.trim_transmat.sum(axis=0)
print(f"Trimmed transition matrix column sums: min = {col_sums_trim.min():.6f}, max = {col_sums_trim.max():.6f}")

The `trim_indices` list maps each position in the trimmed matrix back to its
original state index.

In [ ]:
print(f"First 10 trim_indices: {our_csn.trim_indices[:10]}")

## 5. Steady-state weights

The steady-state (equilibrium) population of each state can be estimated two ways.

### Eigenvalue method
The leading eigenvector of $T$ (eigenvalue = 1) gives the stationary distribution.
This is fast but can be numerically imprecise for low-probability states.

In [ ]:
wt_eig = our_csn.calc_eig_weights()
print(f"Eigenvalue weights: sum = {wt_eig.sum():.6f}, min = {wt_eig.min():.2e}, max = {wt_eig.max():.4f}")

### Iterative multiplication method
Repeatedly squaring $T$ until convergence ($T^{2^k} \to T^\infty$ as $k \to \infty$).
More numerically stable for near-degenerate matrices, but requires the matrix to be
**aperiodic**.  After trimming, self-transitions are added automatically which
breaks any periodicity.

In [ ]:
wt_mult = our_csn.calc_mult_weights()
print(f"Mult weights: sum = {wt_mult.sum():.6f}, min = {wt_mult.min():.2e}, max = {wt_mult.max():.4f}")

In [ ]:
# The two methods should agree closely
fig, ax = plt.subplots(figsize=(5, 5))
max_wt = max(wt_eig.max(), wt_mult.max())
ax.scatter(wt_eig, wt_mult, alpha=0.3, s=4, color='steelblue')
ax.plot([0, max_wt], [0, max_wt], 'r-', lw=1, label='y = x')
ax.set_xlabel("Eigenvalue weight")
ax.set_ylabel("Iterative multiplication weight")
ax.set_title("Steady-state weight comparison")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Node attributes

Computed quantities are automatically stored as node attributes on `our_csn.graph`,
making them available for any NetworkX analysis or export.

In [ ]:
node_id = our_csn.trim_indices[0]
print(f"Attributes stored on node {node_id}:")
for k, v in our_csn.graph.nodes[node_id].items():
    print(f"  {k:15s}: {v}")

In [ ]:
# Top 10 states by equilibrium weight
top_states = np.argsort(wt_eig)[::-1][:10]
print("Top 10 states by equilibrium weight:")
print(f"  {'State':>6}  {'Weight':>10}")
for s in top_states:
    print(f"  {s:6d}  {wt_eig[s]:10.6f}")

---
**Next:** [02_committors_and_visualization.ipynb](02_committors_and_visualization.ipynb) —
computing committor probabilities and exporting to Gephi.